**Laboratorio de Métodos Cuantitativos Aplicados a la Gestión**

---

# **Clase 01 - Manejo de archivos y obtención de datos organizacionales**

## Complemento

Este notebook se complementa con la presentación: **DATA.pdf**

Te recomendamos leer el PDF para trabajar con este notebook y tener una mejor comprensión de los conceptos teóricos.

## ¿Qué vamos a hacer en esta clase?

Toda la materia se apoya en un gesto que vamos a repetir cien veces: **traer datos a Python y mirarlos**.
Hoy aprendemos a hacerlo bien, con datos de una organización real.

| Parte | Tema | Qué nos llevamos |
|---|---|---|
| **A** | ¿De dónde salen los datos? | Los 3 formatos que te vas a cruzar siempre |
| **B** | El problema de las rutas | Por qué a tu compañero le anda y a vos no |
| **C** | Cargar: CSV, Excel y JSON | `read_csv`, `read_excel`, `read_json` |
| **D** | Primer contacto con la tabla | `shape`, `head`, `info`, `describe` |
| **E** | Seleccionar y filtrar | Quedarse con lo que importa |
| **F** | Guardar el resultado | `to_csv`, `to_excel` |

> **Al terminar** tenés que poder agarrar un archivo cualquiera, cargarlo y decir en voz alta
> cuántas filas tiene, qué mide cada columna y dónde faltan datos.

Arrancamos importando las librerías. Esta celda va a estar al principio de **todos** los notebooks de la materia.

In [ ]:
import pandas as pd               # pandas: la librería para trabajar con tablas
import numpy as np                # numpy: cálculo numérico
import matplotlib.pyplot as plt   # matplotlib: gráficos

pd.set_option("display.float_format", "{:,.2f}".format)   # que los números se vean con 2 decimales y separador de miles

---
# 📁 Parte A — ¿De dónde salen los datos?

En una organización los datos casi nunca llegan en un formato lindo. Llegan así:

| Formato | Extensión | De dónde suele venir | Cómo se lee |
|---|---|---|---|
| **CSV** | `.csv` | Exportación de un sistema, ERP, Google Sheets | `pd.read_csv()` |
| **Excel** | `.xlsx` `.xls` | Alguien de administración, con colores y celdas combinadas | `pd.read_excel()` |
| **JSON** | `.json` | Una API, un sistema web | `pd.read_json()` |

**CSV** significa *Comma Separated Values*: es un archivo de texto plano donde cada línea es una fila
y las columnas van separadas por comas. Si lo abrís con el Bloc de notas, lo podés leer. Esa es toda la magia.

```
Fecha,Producto,Vendedor,Ciudad,Cantidad,Precio_Unitario
2024-01-01,Monitor,Juan,Mendoza,6,23030
2024-01-02,Notebook,Lucía,La Plata,7,87838
```

> ⚠️ **Ojo con el CSV en castellano.** En Argentina Excel a veces usa `;` como separador y `,` como
> decimal. Si al cargar te queda **todo en una sola columna**, probá `pd.read_csv(ruta, sep=";")`.

---
# 🗺️ Parte B — El problema de las rutas

Esta es **la** causa número uno de que un notebook no ande. Python no adivina dónde está tu archivo:
hay que decírselo con una **ruta**.

| Tipo de ruta | Ejemplo | Problema |
|---|---|---|
| **Absoluta** | `C:\Users\juan\Escritorio\ventas.csv` | Solo funciona en **tu** computadora |
| **Relativa** | `../DF/ventas.csv` | Funciona si respetás la estructura de carpetas |
| **URL** | `https://raw.githubusercontent.com/.../ventas.csv` | ✅ Funciona en todos lados |

En esta materia usamos **siempre la URL**: así el notebook corre igual en Colab, en tu casa y en la facultad,
sin que nadie tenga que editar nada ni montar Google Drive.

In [ ]:
# Esta variable la vamos a reutilizar en todas las clases.
# Apunta a la carpeta DF/ del repositorio de la materia en GitHub.
URL = "https://raw.githubusercontent.com/Datso653/Laboratorio-de-metodos-Cuantitativos-Aplicados-a-la-gestion/main/DF/"

print(URL + "ventas.csv")   # así se arma la dirección de cada archivo

---
# 📥 Parte C — Cargar los datos

## C1. Un CSV

Vamos a trabajar con las **ventas de una empresa de tecnología**: 200 operaciones con fecha, producto,
vendedor, ciudad, cantidad y precio.

In [ ]:
ventas = pd.read_csv(URL + "ventas.csv")   # read_csv: leer un archivo CSV y devolver un DataFrame

ventas.head()   # head(): muestra las primeras 5 filas

Lo que acabamos de crear se llama **DataFrame**: una tabla con filas y columnas, como una hoja de Excel
pero manejable con código.

- Cada **fila** es una observación (acá: una venta).
- Cada **columna** es una variable (fecha, producto, cantidad...).
- El número gris de la izquierda es el **índice**: el nombre de cada fila. Arranca en **0**, no en 1.

## C2. Un Excel

Cambia una sola palabra: `read_excel` en lugar de `read_csv`. Usamos las notas de la cursada de 2024.

In [ ]:
notas = pd.read_excel(URL + "notas_2024.xlsx")   # read_excel: leer una planilla de Excel

notas.head()

> 📌 **Si el Excel tiene varias hojas**, agregás el argumento `sheet_name`:
> ```python
> pd.read_excel(ruta, sheet_name="Ventas2024")   # por nombre
> pd.read_excel(ruta, sheet_name=0)              # o por posición (0 = la primera)
> ```
>
> **Si el Excel tiene un título arriba** y la tabla arranca más abajo, usás `skiprows`:
> ```python
> pd.read_excel(ruta, skiprows=3)   # ignorar las primeras 3 filas
> ```

## C3. Un JSON

Los datos que vienen de una **API** (un sistema que te entrega información por internet) suelen llegar
en formato JSON. Se lee con `read_json`.

In [ ]:
# lines=True porque este archivo tiene un registro JSON por línea
monitores = pd.read_json(URL + "ventas_monitores.json", lines=True)

monitores.head(3)

Fijate en la columna `Fecha`: son números larguísimos. Eso pasa porque el JSON guarda las fechas como
*milisegundos desde 1970* (el "tiempo Unix"). Se arregla con una línea:

In [ ]:
# to_datetime: convertir una columna a formato fecha. unit="ms" avisa que el número está en milisegundos
monitores["Fecha"] = pd.to_datetime(monitores["Fecha"], unit="ms")

monitores.head(3)

---
# 🔍 Parte D — Primer contacto con la tabla

Ya tenemos los datos adentro. **Antes de calcular nada**, hay que mirarlos. Siempre estas cuatro preguntas:

| Pregunta | Comando |
|---|---|
| ¿Cuán grande es? | `df.shape` |
| ¿Qué pinta tiene? | `df.head()` |
| ¿De qué tipo es cada columna y dónde faltan datos? | `df.info()` |
| ¿Cómo se distribuyen los números? | `df.describe()` |

In [ ]:
# shape: devuelve (cantidad de filas, cantidad de columnas)
print("Dimensiones:", ventas.shape)
print("Filas:", ventas.shape[0], "| Columnas:", ventas.shape[1])

In [ ]:
# info(): tipo de dato de cada columna y cuántos valores NO nulos tiene
ventas.info()

**Cómo se lee un `info()`:**

- `Non-Null Count` → cuántas filas tienen dato. Si dice menos que el total, **hay datos faltantes**.
- `Dtype` → el tipo:

| Dtype | Significa | Cuidado |
|---|---|---|
| `int64` | Número entero | |
| `float64` | Número con decimales | |
| `object` | Texto (o mezcla) | ⚠️ Si una columna numérica figura como `object`, hay algún texto colado |
| `datetime64` | Fecha | Hay que convertirla, no viene sola |

Fijate que `Fecha` figura como `object`: pandas la leyó como **texto**, no como fecha. Lo arreglamos:

In [ ]:
ventas["Fecha"] = pd.to_datetime(ventas["Fecha"])   # convertir el texto a fecha de verdad

ventas.dtypes   # dtypes: ver el tipo de cada columna

In [ ]:
# describe(): estadísticas de las columnas numéricas
ventas.describe().round(2)

Ese cuadro ya dice bastante del negocio:

- `count` → cuántos datos hay.
- `mean` → el promedio.
- `std` → el desvío estándar: qué tan dispersos están los valores.
- `min` y `max` → el mínimo y el máximo.
- `25%`, `50%`, `75%` → los cuartiles. El `50%` es la **mediana**: el valor del medio.

> 💡 Si el **promedio** y la **mediana** están muy lejos uno del otro, hay valores extremos tirando del promedio.
> Lo vamos a ver en detalle en la clase de métricas estadísticas.

---
# ✂️ Parte E — Seleccionar y filtrar

Rara vez usamos la tabla entera. Estas cuatro operaciones cubren el 90% de los casos.

### Una columna

In [ ]:
ventas["Producto"].head()          # corchetes con el nombre de la columna entre comillas

### Varias columnas — ojo con el **doble** corchete

`ventas["Producto"]` te da **una** columna. Para pedir varias, le pasás una **lista** de nombres,
y la lista también va entre corchetes. De ahí el `[[ ]]`.

In [ ]:
ventas[["Producto", "Cantidad", "Precio_Unitario"]].head()

### Crear una columna nueva

Acá es donde los datos empiezan a servir. Tenemos cantidad y precio unitario, pero **no** el total facturado.
Lo calculamos:

In [ ]:
ventas["Total"] = ventas["Cantidad"] * ventas["Precio_Unitario"]

ventas.head()

### Filtrar filas

Le pasamos una **condición** entre corchetes. Python evalúa la condición fila por fila y se queda
solo con las que dan verdadero.

In [ ]:
# Solo las ventas de Córdoba
cordoba = ventas[ventas["Ciudad"] == "Córdoba"]

print("Ventas totales:", len(ventas))
print("Ventas en Córdoba:", len(cordoba))
cordoba.head()

In [ ]:
# Dos condiciones a la vez:
#   &  significa "Y" (se tienen que cumplir las dos)
#   |  significa "O" (alcanza con una)
# Cada condición va entre paréntesis; no es opcional.
grandes = ventas[(ventas["Ciudad"] == "Córdoba") & (ventas["Total"] > 200_000)]

print("Ventas grandes en Córdoba:", len(grandes))
grandes.head()

### Agrupar: la pregunta del jefe

*"¿Cuánto vendió cada vendedor?"* En Excel sería una tabla dinámica. Acá es una línea: `groupby`.

In [ ]:
# groupby("Vendedor") → armar un grupo por cada vendedor
# ["Total"].sum()     → sumar la columna Total dentro de cada grupo
# sort_values(...)    → ordenar de mayor a menor
por_vendedor = ventas.groupby("Vendedor")["Total"].sum().sort_values(ascending=False)

por_vendedor

In [ ]:
# Y lo mismo en un gráfico, para que se entienda de un vistazo
fig, ax = plt.subplots(figsize=(8, 4))
por_vendedor.plot(kind="bar", ax=ax, color="#243b5e")
ax.set_title("Facturación total por vendedor", loc="left", fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("Total facturado ($)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

---
# 💾 Parte F — Guardar el resultado

El trabajo no termina en la pantalla: casi siempre hay que devolver un archivo.

In [ ]:
# index=False evita que se guarde el número de fila como si fuera una columna más
ventas.to_csv("ventas_con_total.csv", index=False)

print("Archivo guardado ✅")

> 📌 **En Colab**, el archivo queda en la máquina virtual (panel de carpetas 📁 a la izquierda) y
> **se borra al cerrar la sesión**. Para bajarlo a tu computadora:
> ```python
> from google.colab import files
> files.download("ventas_con_total.csv")
> ```
>
> Para guardar en Excel es `to_excel` en lugar de `to_csv`:
> ```python
> ventas.to_excel("ventas_con_total.xlsx", index=False)
> ```

---
# 📝 Ejercicios

Trabajamos con `productos_stock.csv`: el inventario de una distribuidora con 10 productos.

In [ ]:
stock = pd.read_csv(URL + "productos_stock.csv")
stock

**Ejercicio 1.** ¿Cuántas filas y columnas tiene `stock`? ¿Qué tipo de dato tiene cada columna?

In [ ]:
# Tu respuesta acá

**Ejercicio 2.** Creá una columna `Valor_inventario` = `Stock actual` × `Precio unitario (€)`.
¿Cuál es el producto con más capital inmovilizado?

In [ ]:
# Tu respuesta acá

**Ejercicio 3.** Filtrá los productos con más de 30 días en stock. Son los que se están quedando
dormidos en el depósito: ¿cuánto dinero representan en total?

In [ ]:
# Tu respuesta acá

**Ejercicio 4.** Usando `groupby`, calculá las ventas mensuales totales por `Categoría`
y hacé un gráfico de barras.

In [ ]:
# Tu respuesta acá

**Ejercicio 5 (integrador).** Cargá `ventas_monitores.csv` y compará su cantidad de filas con la
del `ventas.csv` filtrado solo por monitores. ¿Dan lo mismo? ¿Qué te dice eso sobre los dos archivos?

In [ ]:
# Tu respuesta acá

---
## 🧭 Para la próxima

| Concepto | Comando |
|---|---|
| Cargar CSV / Excel / JSON | `pd.read_csv()` · `pd.read_excel()` · `pd.read_json()` |
| Tamaño de la tabla | `df.shape` |
| Primeras filas | `df.head()` |
| Tipos y faltantes | `df.info()` |
| Estadísticas | `df.describe()` |
| Una columna / varias | `df["col"]` · `df[["a", "b"]]` |
| Filtrar | `df[df["col"] > valor]` |
| Agrupar | `df.groupby("col")["otra"].sum()` |
| Guardar | `df.to_csv("archivo.csv", index=False)` |

En la próxima clase (**21-ago, Visualización**) usamos este mismo DataFrame de ventas para aprender a
contar la historia con gráficos.